# 🧠 MockMate AI — Train Your Custom Model
**Just click Runtime → Run All. That's it!**

In [ ]:
# Step 1: Install (takes ~3 min)
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers

In [ ]:
# Step 2: Upload your training data
from google.colab import files
import json, os

print('📂 Upload your training_*.jsonl file:')
uploaded = files.upload()
TRAIN_FILE = list(uploaded.keys())[0]

data = []
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                data.append(json.loads(line))
            except:
                pass

print(f'✅ Loaded {len(data)} training samples')
print(f'📝 Sample: {json.dumps(data[0], indent=2)[:300]}...')

In [ ]:
# Step 3: Load model (4-bit to save VRAM)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/tinyllama-bnb-4bit',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('✅ Model loaded!')

In [ ]:
# Step 4: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)
print('✅ LoRA adapters added!')

In [ ]:
# Step 5: Format data
from datasets import Dataset

PROMPT = '''Below is an instruction that describes a task, paired with an input. Write a response.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}'''

def fmt(s):
    return {'text': PROMPT.format(
        instruction=s.get('instruction',''),
        input=s.get('input',''),
        output=s.get('output','')
    ) + tokenizer.eos_token}

dataset = Dataset.from_list(data).map(fmt)
print(f'✅ Formatted {len(dataset)} samples')

In [ ]:
# Step 6: Train! (~15-20 min)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir='outputs',
        optim='adamw_8bit',
        seed=42,
    ),
)

print('🚀 Training started...')
stats = trainer.train()
print(f'\n✅ Done! Loss: {stats.training_loss:.4f}, Time: {stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# Step 7: Test it!
FastLanguageModel.for_inference(model)

test = PROMPT.format(
    instruction='Generate a Medium difficulty dsa interview question.',
    input='Data Structures & Algorithms, Medium',
    output=''
)

inputs = tokenizer(test, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
result = tokenizer.decode(out[0], skip_special_tokens=True)
print('🧪 Test:', result.split('### Response:')[-1].strip()[:500])

In [ ]:
# Step 8: Save & Download
model.save_pretrained_gguf('mockmate-model', tokenizer, quantization_method='q4_k_m')

import glob
gguf = glob.glob('mockmate-model/*.gguf')
if gguf:
    print(f'📥 Downloading {gguf[0]}...')
    files.download(gguf[0])
    print('🎉 Save to F:\\MockMate-AI-Training\\models\\')
else:
    print('❌ GGUF not found. Saving LoRA instead...')
    model.save_pretrained_merged('mockmate-model-merged', tokenizer, save_method='merged_16bit')
    !zip -r mockmate-model.zip mockmate-model-merged/
    files.download('mockmate-model.zip')
    print('🎉 Downloaded! Extract to F:\\MockMate-AI-Training\\models\\')